# Clustering Walkthrough: Wholesale Customers

The last of dscompanion's three supported tasks: unsupervised clustering, using the
UCI **Wholesale Customers** dataset — annual spending by 440 clients of a wholesale
distributor across six product categories (fresh food, milk, grocery, frozen, detergents
& paper, delicatessen), plus two grouping columns (`Channel`: Horeca vs. Retail; `Region`).

There's no target to predict here — we're looking for structure in the spending patterns
themselves. That makes the config slightly different from `02`/`03` in one specific way,
explained below. See [`README.md`](README.md) for the full notebook index.

In [1]:
!pip install -q dscompanion[notebooks]

zsh:1: no matches found: dscompanion[notebooks]


In [2]:
from pathlib import Path
import pandas as pd
from ucimlrepo import fetch_ucirepo

cache_dir = Path(".data_cache")
cache_dir.mkdir(exist_ok=True)
data_path = cache_dir / "wholesale_customers.parquet"

if not data_path.exists():
    ds = fetch_ucirepo(id=292)
    df = ds.data.features.copy()
    df["Region"] = ds.data.targets["Region"]
    df.to_parquet(data_path, index=False)
else:
    df = pd.read_parquet(data_path)

print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df.head()

440 rows, 8 columns


,Channel,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Region
0,2,12669,9656,7561,214,2674,1338,3
1,2,7057,9810,9568,1762,3293,1776,3
2,2,6353,8808,7684,2405,3516,7844,3
3,1,13265,1196,4221,6404,507,1788,3
4,2,22615,5410,7198,3915,1777,5185,3


## A config quirk worth understanding

`DataConfig.target` is a **required** field in dscompanion, even for clustering — but
clustering's evaluation never actually uses it as a supervised label. `ClusteringModel`
computes only *internal* validity metrics (silhouette score, Davies-Bouldin index,
Calinski-Harabasz index) directly from the feature matrix — there's no ground truth to
compare against, by definition of what clustering is.

So here we point `target` at `Channel` purely to satisfy the schema (it has to name
*some* column), and explicitly list the real spending features via `feature_columns` so
that `Channel` and `Region` never leak in as clustering inputs — they're grouping labels,
not spending measurements, and would distort the clusters if left in.

In [3]:
from dscompanion.pipeline import PipelineConfig, PipelineRunner

cfg = PipelineConfig(
    name="wholesale_customers_clustering",
    data={
        "path": str(data_path),
        "format": "parquet",
        "target": "Channel",
        "feature_columns": ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"],
    },
    model={"task": "clustering", "algorithm": "kmeans", "params": {"n_clusters": 3}},
    reporting={"output_dir": "reports", "html_report": True},
)
result = PipelineRunner(cfg).run()
result.metrics.pivot(index="metric", columns="split", values="value")

2026-09-13 11:42:44  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:42:44  INFO      dscompanion.pipeline.runner  PipelineRunner  |  wholesale_customers_clustering  v1.0


2026-09-13 11:42:44  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: clustering  |  Algorithm: kmeans


2026-09-13 11:42:44  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:42:44  WARNING   dscompanion.pipeline.runner  Pre-flight: target.imbalance.strategy='class_weight' is only applicable to classification tasks. It will be ignored.


...

2026-09-13 11:42:45  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:42:45  INFO      dscompanion.pipeline.runner  Pipeline complete — 1.2s  |  Run: 20260913_114244


2026-09-13 11:42:45  INFO      dscompanion.pipeline.runner  Report: reports/20260913_114244/reports/wholesale_customers_clustering_v1.0_model_card.html


2026-09-13 11:42:45  INFO      dscompanion.pipeline.runner  ============================================================


split,train
metric,
calinski_harabasz_score,1.658730e+02
davies_bouldin_score,9.036150e-01
inertia,4.430274e+10
silhouette_score,4.196780e-01


Same tidy `split`/`metric`/`value` shape as classification and regression, but with
one deliberate difference: `ClusteringModel.evaluate()` only ever reports a `"train"`
row — validation, test, and OOT partitions are not evaluated (by design, not a
limitation you're hitting by accident; see the `evaluate()` docstring). Internal
validity metrics like silhouette score measure how well-separated *this specific set of
points* clustered — there's no "held-out generalization" question analogous to
predicting an unseen label, so dscompanion doesn't manufacture one. The test split still
exists on `result.split` (we'll use it below), just not for this metrics table.

For clustering, `metric` can take up to four values:

- `silhouette_score` — how well-separated the clusters are (higher is better, range -1 to 1)
- `davies_bouldin_score` — average similarity between clusters (lower is better)
- `calinski_harabasz_score` — ratio of between-cluster to within-cluster dispersion (higher is better)
- `inertia` — sum of squared distances to the nearest cluster center (only present for
  estimators that expose `inertia_`, which `kmeans` does)

## Teaching moment: do the clusters mean anything?

We deliberately excluded `Channel` from the features the clustering algorithm saw. Now
let's check, purely out of curiosity, whether the clusters KMeans found from spending
patterns alone happen to line up with `Channel` (Horeca vs. Retail) — a grouping the
algorithm never had access to.

There's no "right answer" here: alignment would suggest spending patterns strongly
reflect business type; no alignment would suggest spending varies for other reasons.
Either result is a genuinely interesting observation about unsupervised learning — the
clusters are shaped entirely by the data's own structure, not by what we hoped to find.

In [4]:
test_rows = result.split.X_test
cluster_labels = result.model.predict(test_rows)

comparison = pd.DataFrame({
    "cluster": cluster_labels,
    "Channel": df.loc[test_rows.index, "Channel"].values,
})
pd.crosstab(comparison["cluster"], comparison["Channel"])

Channel,1,2
cluster,,
0,2,17
1,10,2
2,48,9


## Wrap-up

That's all three of dscompanion's supported tasks covered across these four notebooks:
classification (`01`/`02`), regression (`03`), and clustering (`04`) — each against a
real, different UCI dataset. See [`README.md`](README.md) for where to go next, or head
back to `02_classification_bank_marketing.ipynb` for the deepest tour of the library's
explainability and model-card features.